# Train the Hangman BiLSTM on Kaggle GPU

Clones the `approach/bilstm` branch of the project repo and runs training there --
the model segfaults on this machine's CPU-only PyTorch build locally (a Windows
OpenMP/MKL threading conflict), and training is much faster on GPU anyway.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/bilstm"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Train

Char-level BiLSTM trained with a masked-language-model objective: randomly
mask letters in each training word, predict the true letter at each masked
position from bidirectional context. At inference this becomes: feed the
real board mask through the model, sum per-position letter probabilities
across all blanks, guess the highest-scoring unguessed letter.

Bump `--epochs` up now that we're on GPU -- 6 was chosen for a slow CPU
sanity check, not a converged run.

In [ ]:
!python src/train_bilstm.py --epochs 20

## Validate

Same methodology as the classical candidate-filtering + n-gram approach
(`approach/candidate-ngram` branch), for a fair comparison: hold out 10% of
train.txt, play full interactive games against words the model never
trained on.

In [ ]:
!python src/validate_bilstm.py

## Generate submission.csv

Plays the actual game against every word in test.txt using the model
still in this session (no need to save/reload weights first), following
the competition's blueprint exactly: terminate the instant a 6th wrong
guess lands or the word is solved, join the guesses made into one flat
string per word. This is the *sandbox* leaderboard submission -- per the
competition's Final Judgement policy, final hiring decisions re-run the
submitted model/notebook against a separate private word list, so treat
this CSV as a dev checkpoint, not the actual deliverable.

250,000 words will take a while even on GPU (inference is one word/game at
a time, not batched) -- this prints progress every 20,000 words with an
ETA so you can gauge it.

In [ ]:
!python src/generate_submission_bilstm.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes -- copy both the trained weights and
the generated submission there explicitly in case the working directory
changes.

In [ ]:
import shutil
shutil.copy("src/bilstm_masker.pt", "/kaggle/working/bilstm_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved bilstm_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")